# 02. 탐색적 자료 분석 (EDA)

기계학습기초 4조 · 서울 아파트 매매 실거래 2017-01 ~ 2026-10

이 노트북은 **계산하지 않는다.** `python src/eda/run_eda.py`가 만든 표(`reports/eda/tables/`)와 그림(`reports/eda/figures/`)을 읽어 보여 주기만 한다.
숫자를 바꾸려면 `src/eda/`를 고치고 다시 실행한다. 자세한 해석은 [`reports/eda/EDA_report_ko.md`](../reports/eda/EDA_report_ko.md)에 있다.

- 기술 통계다. 가격은 y의 재료라 모델 특성이 아니다.
- 국면 경계는 `config/regime.yaml`의 확정값을 그대로 읽었다.
- 결과는 연관 관계이며 인과로 읽지 않는다.

In [1]:
import json, pathlib
import pandas as pd
from IPython.display import display

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'reports').exists() else pathlib.Path.cwd().parent
EDA = ROOT / 'reports/eda'
S = json.loads((EDA / 'summary.json').read_text(encoding='utf-8'))
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = 30

def T(name):
    return pd.read_csv(EDA / 'tables' / f'{name}.csv', encoding='utf-8-sig')

pd.Series(S['데이터'], name='값').to_frame()

,값
파일,data/final/아파트_실거래_전처리_2017_2026_v5.csv.xz
행,617592
열,43
첫_계약일,2017-01-01
끝_계약일,2026-10-30
자치구_수,25
법정동_수,338
단지_수,9011
계약월_수,118


## 1. 자료 품질

### 1.1 연도·국면별 거래 건수

![월별 거래 건수](../reports/eda/figures/01_월별_거래건수_국면.png)

In [2]:
T('01_건수_연도별')

,계약연도,거래건수,비율(%)
0,2017,104928,16.99
1,2018,81579,13.21
2,2019,74998,12.14
3,2020,80858,13.09
4,2021,41951,6.79
5,2022,12047,1.95
6,2023,34034,5.51
7,2024,55184,8.94
8,2025,77361,12.53
9,2026,54652,8.85


### 1.2 결측

![열별 결측률](../reports/eda/figures/02_열별_결측률.png)

빈 값의 이유는 `결측사유` 열에 적혀 있다. 항목별로 나누면:

In [3]:
display(T('01_결측사유_항목별').head(12))
T('01_결측_사유없는건수')   # 값이 비었는데 사유가 없는 건수 → 모두 0

,항목,이유,건수,비율(%)
0,동,원자료에 동 미공개,424259,68.70
1,난방·복도·시공사,단지 목록에 없어 미기재(건축물대장에는 항목 없음),214475,34.73
2,사용승인일,건축물대장에 사용승인일 미기재(연식은 건축년도 기준),53954,8.74
3,주차,건축물대장에 주차대수 미기재,38961,6.31
4,주차,원자료에 값이 없거나 0으로 기재,29177,4.72
5,건축물대장,채우지 못함: 응답 없음,5033,0.81
6,단지정보,단지정보 없음(단지 목록에서 이 단지를 찾지 못함),4729,0.77
7,단지정보,단지정보 불확실(같은 단지인지 확정하지 못함),1641,0.27
8,건축물대장,채우지 못함: 사용승인연도 불일치,919,0.15
9,주차,기간 중 변경 이력이 의심되어 제외,738,0.12


,열,결측건수,사유_없는_결측
0,동,424259,0
1,세대수,6795,0
2,세대당_주차대수,75555,0
3,난방방식,217716,0
4,위도,358,0


### 1.3 중복

완전 중복과 거래ID 중복은 없다. 단지·면적·층·계약일·금액이 모두 같은 거래 묶음은 지우지 않고 사실만 기록한다.

In [4]:
display(pd.Series({k: v for k, v in S['중복'].items() if k != '같은조건_기준열'}, name='건수').to_frame())
display(pd.Series(S['같은조건_거래'], name='값').to_frame())
T('01_중복_연도별')

,건수
전체열_완전중복_행,0
거래ID_중복_행,0
같은조건_거래_묶음에_속한_행,6997
같은조건_거래_중_두번째_이후_행,4506


,값
묶음_수,"2,491.00"
묶음_최대_크기,14.00
2건_묶음,"1,783.00"
전용40㎡이하_비율(%)_묶음내,65.99
전용40㎡이하_비율(%)_전체,7.94
동_공개된_묶음,967.00
동이_모두_같은_묶음,856.00
동이_다른_묶음,111.00


,계약연도,거래건수,같은조건_두번째_이후,비율(%)
0,2017,104928,269,0.26
1,2018,81579,287,0.35
2,2019,74998,393,0.52
3,2020,80858,397,0.49
4,2021,41951,314,0.75
5,2022,12047,480,3.98
6,2023,34034,475,1.40
7,2024,55184,397,0.72
8,2025,77361,712,0.92
9,2026,54652,782,1.43


### 1.4 값 점검 (0이 아닌 항목만)

문서에 없던 문제: 세대당 주차대수가 **0**으로 적힌 거래, 날짜 형식이 아닌 사용승인일.

In [5]:
display(T('01_값점검').query('위반건수 > 0').fillna(''))
T('01_주차0_단지')

,점검,위반건수,비고
7,층 ≤ 0,120,지하·반지하 표기일 수 있음(오류로 단정하지 않음)
13,건축년도 > 계약연도,39,준공 전 거래(분양권·입주권) 가능
15,연식_기준 = '준공 전 거래',253,config/target.yaml의 제외 대상
16,사용승인일이 날짜 형식(YYYY-MM-DD)이 아님,17,일(日)이 '00'으로 적힌 값
17,사용승인일이 계약일보다 뒤,346,'준공 전 거래' + 같은 달 안에서 승인일이 며칠 늦은 거래
20,좌표 없음,358,결측사유 '주소 소실 추정'. 모델 표본 제외 대상
26,세대당_주차대수 = 0,4550,값은 있으나 0. 결측사유에 주차 항목 없음 → 사실상 미기재로 보임
28,세대당_주차대수 > 3,3869,주상복합 등 비주거 주차 포함 추정(docs/data.md §6)
29,세대수_기준 = '건축물대장(번지 합계)',17592,번지 전체 세대수


,자치구,법정동,번지,단지명,거래건수,세대수,세대수_기준,건축년도
0,송파구,잠실동,35,트리지움,1124,3696,단지 목록(K-apt),2007
1,양천구,신월동,987-1,신월시영,951,2256,단지 목록(K-apt),1988
2,강남구,개포동,141,개포주공1단지,511,5040,단지 목록(K-apt),1982
3,양천구,목동,902,목동신시가지2,407,1640,단지 목록(K-apt),1986
4,중랑구,신내동,618,동성3,308,831,건축물대장,1997
5,노원구,상계동,1110,두산,289,763,단지 목록(K-apt),1994
6,용산구,이촌동,415,한강(대우),267,834,단지 목록(K-apt),2000
7,용산구,이촌동,400,현대한강,163,516,단지 목록(K-apt),1997
8,노원구,상계동,1265,수락산벨리체,149,492,단지 목록(K-apt),2000
9,마포구,중동,390,DMC마포청구아파트,114,420,단지 목록(K-apt),1994


### 1.5 분위수와 이상치

꼬리를 보는 용도다. 지울 대상이 아니다.

In [6]:
display(T('01_분위수_요약'))
display(T('01_이상치_IQR'))
pd.Series(S['가격_지역내_이상치'], name='값').to_frame()

,열,건수,평균,최솟값,p0.1,p1,p5,p25,중앙값,p75,p95,p99,p99.9,최댓값
0,거래금액_만원,617592,"91,921.60","4,732.00","8,800.00","12,500.00","24,900.00","47,000.00","72,000.00","113,500.00","225,000.00","374,000.00","687,045.00","2,900,000.00"
1,㎡당가_만원,617592,"1,191.19",121.34,280.01,363.99,464.48,703.67,999.10,"1,453.41","2,588.51","3,928.53","5,653.39","10,586.70"
2,전용면적(㎡),617592,76.76,10.02,12.11,14.46,34.44,59.68,82.32,84.96,131.44,173.17,243.38,395.06
3,층,617592,9.34,-3.00,1.00,1.00,1.00,4.00,8.00,13.00,21.00,28.00,43.00,69.00
4,건축년도,617592,"2,001.34","1,961.00","1,970.00","1,976.00","1,985.00","1,994.00","2,001.00","2,008.00","2,019.00","2,023.00","2,026.00","2,026.00"
5,건물연식,617553,19.55,0.00,0.00,0.00,3.00,12.00,19.00,27.00,37.00,45.00,53.00,65.00
6,연식_개월,617339,233.76,0.00,0.00,3.00,35.00,141.00,231.00,321.00,448.00,541.00,634.00,775.00
7,세대수,610797,"1,039.05",2.00,6.00,14.00,42.00,268.00,660.00,"1,372.00","3,544.00","5,563.00","9,510.00","12,032.00"
8,세대당_주차대수,542037,1.09,0.00,0.00,0.01,0.36,0.81,1.10,1.26,1.81,2.85,5.40,12.46
9,최근접_역_거리(m),617234,555.50,18.00,46.00,90.00,163.00,323.00,478.00,691.00,"1,243.00","1,867.00","2,480.00","3,595.00"


,열,하한,상한,하한_미만,상한_초과,이상치_비율(%)
0,거래금액_만원,"-52,750.00","213,250.00",0,35052,5.68
1,㎡당가_만원,-420.94,"2,578.02",0,31351,5.08
2,전용면적(㎡),21.75,122.89,16743,40642,9.29
3,층,-9.50,26.50,0,8875,1.44
4,건축년도,"1,973.00","2,029.00",2436,0,0.39
5,건물연식,-10.50,49.50,0,1646,0.27
6,연식_개월,-129.00,591.00,0,1705,0.28
7,세대수,"-1,388.00","3,028.00",0,38648,6.33
8,세대당_주차대수,0.14,1.94,7592,21041,5.28
9,최근접_역_거리(m),-229.00,"1,243.00",0,30727,4.98


,값
기준,"자치구×계약월(거래 30건 이상) 안 ln(㎡당가)의 robust z, |z| > 3.0"
판정가능_거래,614086
낮은쪽,4995
높은쪽,2257
비율(%),1.18


## 2. 가격

![가격 분포](../reports/eda/figures/03_가격_분포.png)

In [7]:
display(T('02_가격_요약'))
T('02_가격대_비중')

,변수,건수,평균,표준편차,p5,p25,중앙값,p75,p95,최솟값,최댓값,왜도
0,거래금액_억원,617592,9.19,7.43,2.49,4.70,7.20,11.35,22.50,0.47,290.00,3.91
1,㎡당가_만원,617592,"1,191.19",725.32,464.48,703.67,999.10,"1,453.41","2,588.51",121.34,"10,586.70",2.16
2,평당가_만원,617592,"3,937.80","2,397.74","1,535.00","2,326.00","3,303.00","4,805.00","8,557.00",401.00,"34,997.00",2.16
3,전용면적(㎡),617592,76.76,30.10,34.44,59.68,82.32,84.96,131.44,10.02,395.06,1.13


,가격대,거래건수,비율(%)
0,3억 이하,53957,8.74
1,3~6억,191243,30.97
2,6~9억,154244,24.98
3,9~15억,134174,21.73
4,15~30억,71264,11.54
5,30억 초과,12710,2.06


### 2.1 자치구

![자치구별 ㎡당 가격](../reports/eda/figures/04_자치구별_㎡당가.png)

In [8]:
T('02_가격_자치구별')

,자치구,거래건수,p5,p25,중앙값,p75,p95,거래금액_억원_중앙값,전용면적_중앙값,권역
0,강남구,33651,936.92,"1,461.76","2,072.29","2,820.41","4,208.36",17.00,84.72,동남권
1,서초구,26508,796.58,"1,170.75","1,742.20","2,602.47","4,338.90",16.50,84.94,동남권
2,송파구,41140,678.39,"1,059.55","1,516.13","2,139.80","3,336.09",12.05,84.79,동남권
3,용산구,12081,694.77,"1,073.88","1,504.05","2,050.77","3,135.34",13.47,84.94,도심권
4,성동구,24932,652.59,988.36,"1,383.25","1,867.29","2,709.56",10.45,84.45,동북권
5,마포구,24182,653.91,941.35,"1,295.64","1,765.33","2,571.79",9.70,84.45,서북권
6,광진구,12598,605.71,884.67,"1,216.87","1,648.61","2,247.07",9.50,84.56,동북권
7,동작구,23738,585.52,865.62,"1,164.42","1,566.14","2,287.12",9.00,84.77,서남권
8,영등포구,28016,578.01,818.72,"1,124.05","1,499.59","2,200.73",8.25,79.78,서남권
9,중구,8571,566.42,835.86,"1,102.03","1,468.01","2,066.64",8.50,83.97,도심권


### 2.2 면적·층·연식

세로축은 같은 계약 연도·같은 자치구의 ㎡당가 중앙값 대비 %다(시기·위치를 걷어낸 상대 수준). 특성끼리 겹치므로 각 특성의 고유 효과가 아니다.

![면적·층·연식](../reports/eda/figures/05_가격_면적_층_연식.png)

In [9]:
cols = ['거래건수', '중앙값', '거래금액_억원_중앙값', '상대가격_중앙값(%)']
for n, k in [('02_가격_면적구간별', '면적구간'), ('02_가격_층구간별', '층구간'), ('02_가격_연식구간별', '연식구간')]:
    display(T(n)[[k] + cols])
T('02_가격_순위상관')

,면적구간,거래건수,중앙값,거래금액_억원_중앙값,상대가격_중앙값(%)
0,40㎡ 이하,49031,987.52,2.72,2.65
1,40~60㎡,203919,"1,059.82",5.99,9.12
2,60~85㎡,257123,954.40,7.85,-3.33
3,85~102㎡,16370,"1,088.56",10.50,0.91
4,102~135㎡,66736,895.81,10.50,-14.54
5,135㎡ 초과,24413,"1,164.99",18.70,-13.61


,층구간,거래건수,중앙값,거래금액_억원_중앙값,상대가격_중앙값(%)
0,0층 이하,120,756.83,7.03,-29.89
1,1~2층,72140,889.55,6.48,-8.03
2,3~5층,126936,939.00,6.65,-4.50
3,6~10층,186161,988.35,6.98,-0.35
4,11~15층,142417,"1,017.51",7.25,2.31
5,16~20층,57404,"1,094.34",8.40,6.24
6,21~30층,28231,"1,301.56",10.80,14.70
7,31층 이상,4183,"1,654.65",17.40,23.12


,연식구간,거래건수,중앙값,거래금액_억원_중앙값,상대가격_중앙값(%)
0,5년 이하,65660,"1,343.02",8.90,25.72
1,6~10년,68894,"1,119.09",8.70,13.20
2,11~15년,89075,870.90,7.20,-2.31
3,16~20년,114177,854.70,6.62,-7.02
4,21~25년,102752,935.35,6.80,-8.30
5,26~30년,77260,910.38,6.15,-4.78
6,31~35년,54488,"1,042.04",6.65,2.30
7,36~40년,29826,"1,359.34",9.78,8.99
8,40년 초과,15421,"1,738.56",16.80,10.58


,변수,건수,순위상관_전체,순위상관_연도·자치구_보정
0,전용면적(㎡),617592,-0.00,-0.16
1,층,617592,0.14,0.17
2,건물연식,617553,-0.04,-0.16
3,세대수,610797,0.27,0.48
4,세대당_주차대수,542037,0.18,0.10
5,최근접_역_거리(m),617234,-0.19,-0.14
6,1km내_노선수,617234,0.25,0.11
7,최근접_초등학교_거리(m),617234,-0.01,-0.11
8,1km내_학교수,617234,-0.09,0.09


## 3. 시장 국면

### 3.1 거래 건수와 불균형

![국면별 거래 건수](../reports/eda/figures/06_국면별_거래건수.png)

In [10]:
display(T('03_국면_건수'))
T('03_국면_구분별_건수')

,국면ID,국면,거래건수,자료_개월수,첫_계약월,끝_계약월,구분,비율(%),월평균_거래건수,최다국면_대비
0,up1,상승기①,183144,22,2017-01,2018-10,up,29.65,8325,0.98
1,corr,조정기,11875,6,2018-11,2019-04,down,1.92,1979,0.06
2,up2,상승기②,186805,30,2019-05,2021-10,up,30.25,6227,1.00
3,down,하락기,15948,15,2021-11,2023-01,down,2.58,1063,0.09
4,rec,회복기,68190,18,2023-02,2024-07,recovery,11.04,3788,0.36
5,up3,상승기③,151630,27,2024-08,2026-10,up,24.55,5616,0.81


,구분,거래건수,자료_개월수,비율(%),월평균_거래건수
0,up,521579,79,84.45,6602
1,down,27823,21,4.51,1325
2,recovery,68190,18,11.04,3788


### 3.2 분포 비교

원시 분포다. 국면마다 거래된 집의 구성이 달라 **가격지수로 읽으면 안 된다.**

![국면별 가격·면적 분포](../reports/eda/figures/07_국면별_가격_면적_분포.png)

In [11]:
d = T('03_국면_분포')
display(d[d['변수'].isin(['㎡당가_만원', '전용면적(㎡)', '세대수'])])
display(T('03_국면_KS_㎡당가'))
T('03_국면_KS_전용면적')

,변수,국면ID,국면,건수,평균,p5,p25,중앙값,p75,p95
0,㎡당가_만원,up1,상승기①,183144,791.86,402.03,535.68,681.25,919.89,"1,579.43"
1,㎡당가_만원,corr,조정기,11875,881.18,398.26,567.52,743.18,"1,041.35","1,902.76"
2,㎡당가_만원,up2,상승기②,186805,"1,146.91",502.19,750.50,"1,003.20","1,383.56","2,296.10"
3,㎡당가_만원,down,하락기,15948,"1,369.40",614.19,904.52,"1,183.43","1,605.55","2,832.79"
4,㎡당가_만원,rec,회복기,68190,"1,462.17",672.49,957.85,"1,277.70","1,765.40","2,936.66"
5,㎡당가_만원,up3,상승기③,151630,"1,611.72",695.08,"1,000.35","1,357.95","1,941.63","3,478.14"
12,전용면적(㎡),up1,상승기①,183144,79.93,39.60,59.82,84.42,84.98,134.71
13,전용면적(㎡),corr,조정기,11875,75.29,21.07,56.78,74.05,84.96,136.79
14,전용면적(㎡),up2,상승기②,186805,76.35,34.44,59.58,79.98,84.96,131.86
15,전용면적(㎡),down,하락기,15948,66.99,14.43,41.30,59.97,84.92,133.75


,국면,상승기①,조정기,상승기②,하락기,회복기,상승기③
0,상승기①,0.00,0.09,0.34,0.49,0.54,0.56
1,조정기,0.09,0.00,0.26,0.41,0.46,0.48
2,상승기②,0.34,0.26,0.00,0.17,0.21,0.26
3,하락기,0.49,0.41,0.17,0.00,0.07,0.12
4,회복기,0.54,0.46,0.21,0.07,0.00,0.06
5,상승기③,0.56,0.48,0.26,0.12,0.06,0.00


,국면,상승기①,조정기,상승기②,하락기,회복기,상승기③
0,상승기①,0.00,0.10,0.06,0.19,0.08,0.07
1,조정기,0.10,0.00,0.07,0.12,0.09,0.08
2,상승기②,0.06,0.07,0.00,0.17,0.03,0.02
3,하락기,0.19,0.12,0.17,0.00,0.18,0.17
4,회복기,0.08,0.09,0.03,0.18,0.00,0.01
5,상승기③,0.07,0.08,0.02,0.17,0.01,0.00


### 3.3 거래 구성과 선택 편향

하락기·조정기에는 작은 집·작은 단지의 거래 비중이 커지고, 거래가 일어난 단지 자체가 절반 이하로 줄어든다.

![국면별 거래 구성](../reports/eda/figures/08_국면별_거래구성.png)

In [12]:
display(T('03_국면_거래구성').set_index('국면').drop(columns='국면ID').T)
display(T('03_국면_거래단지수'))
display(T('03_단지_거래된국면수'))
display(T('03_단지_전국면거래_비교'))
T('03_국면_모델제외대상')

국면,상승기①,조정기,상승기②,하락기,회복기,상승기③,전체
동남권(강남·서초·송파·강동) 비중(%),22.40,20.57,20.69,22.68,25.34,21.02,21.84
강남3구 비중(%),17.11,15.88,15.56,17.86,19.18,15.22,16.40
전용 60㎡ 이하 비중(%),36.00,43.33,41.84,53.27,43.86,43.08,40.96
전용 85㎡ 초과 비중(%),20.69,18.29,17.09,14.23,14.87,15.24,17.41
신축(5년 이하) 비중(%),9.77,11.98,10.18,18.34,14.48,9.59,10.63
30년 초과 비중(%),9.92,16.26,15.29,14.70,18.30,23.91,16.15
"1,000세대 이상 비중(%)",33.72,27.77,33.16,27.86,40.28,35.50,34.46
300세대 미만 비중(%),28.20,40.49,30.15,45.19,22.78,25.72,28.24
상위10 브랜드 단지명 비중(%),18.26,13.32,18.71,17.45,28.29,24.42,20.90
토지거래허가구역 거래 비중(%),0.00,0.00,0.71,4.68,4.88,47.44,12.52


,국면ID,국면,거래된_단지수,전체_단지_대비(%),단지당_거래건수_중앙값,거래 1건뿐인 단지 비율(%)
0,up1,상승기①,7298,80.99,6,16.44
1,corr,조정기,3592,39.86,2,44.54
2,up2,상승기②,7844,87.05,6,15.15
3,down,하락기,4138,45.92,2,43.23
4,rec,회복기,5547,61.56,4,27.56
5,up3,상승기③,7003,77.72,6,20.56


,거래된_국면수,단지수,거래건수,단지_비율(%),거래_비율(%)
0,1,804,8725,8.92,1.41
1,2,1186,9006,13.16,1.46
2,3,1652,20337,18.33,3.29
3,4,1682,49083,18.67,7.95
4,5,1560,89255,17.31,14.45
5,6,2127,441186,23.60,71.44


,구분,단지수,거래건수,세대수_중앙값,건축년도_중앙값,㎡당가_중앙값(만원),세대수_결측률(%),난방방식_결측률(%)
0,모든 국면에서 거래된 단지,2127,441186,864,2000,987.17,0.25,28.34
1,일부 국면에서만 거래된 단지,6884,176406,208,2004,"1,037.61",3.21,52.55


,국면ID,국면,좌표 없음,준공 전 거래,둘 중 하나
0,up1,상승기①,332,123,455
1,corr,조정기,8,6,14
2,up2,상승기②,17,96,113
3,down,하락기,1,2,3
4,rec,회복기,0,12,12
5,up3,상승기③,0,14,14
6,all,전체,358,253,611


## 4. 특성 확보율

값은 계약일 기준이다. 모델 특성은 국면 시작 월 a 기준으로 다시 계산된다(`config/features.yaml`).

In [13]:
display(T('04_특성_결측_전체'))
display(T('04_특성_결측_단지단위'))
T('04_세대수_출처별_규모')

,특성,열,값_있음,결측건수,결측률(%),묶음
0,세대수,세대수,610797,6795,1.10,단지 내부
1,세대당 주차대수,세대당_주차대수,542037,75555,12.23,단지 내부
2,난방방식,난방방식,399876,217716,35.25,단지 내부
3,복도유형,복도유형,400102,217490,35.22,단지 내부
4,시공사(상위10 여부),시공사_상위10,399566,218026,35.30,단지 내부
5,사용승인일,사용승인일,557708,59884,9.70,단지 내부
6,최근접 역 거리,최근접_역_거리(m),617234,358,0.06,입지
7,1km 내 노선 수,1km내_노선수,617234,358,0.06,입지
8,최근접 초등학교 거리,최근접_초등학교_거리(m),617234,358,0.06,입지
9,1km 내 학교 수,1km내_학교수,617234,358,0.06,입지


,특성,단지수,값이_한번도_없는_단지,단지_결측률(%),거래_결측률(%)
0,세대수,9011,220,2.44,1.10
1,세대당 주차대수,9011,1077,11.95,12.23
2,난방방식,9011,6925,76.85,35.25
3,복도유형,9011,6925,76.85,35.22
4,시공사(상위10 여부),9011,6928,76.88,35.30
5,사용승인일,9011,631,7.00,9.70
6,최근접 역 거리,9011,37,0.41,0.06
7,1km 내 노선 수,9011,37,0.41,0.06
8,최근접 초등학교 거리,9011,37,0.41,0.06
9,1km 내 학교 수,9011,37,0.41,0.06


,세대수_출처,거래건수,세대수_p25,세대수_중앙값,세대수_p75
0,(세대수 없음),6795,NaN,NaN,NaN
1,건축물대장,196883,78.00,224.00,854.00
2,건축물대장(번지 합계),17592,"1,890.00","3,544.00","3,930.00"
3,단지 목록(K-apt),396322,417.00,791.00,"1,509.00"


### 4.1 국면·연도별

![특성 결측률 — 국면·연도](../reports/eda/figures/09_특성_결측률_국면_연도.png)

In [14]:
T('04_특성_결측_국면별')

,국면ID,국면,거래건수,세대수,세대당 주차대수,난방방식,복도유형,시공사(상위10 여부),사용승인일,최근접 역 거리,1km 내 노선 수,최근접 초등학교 거리,1km 내 학교 수
0,up1,상승기①,183144,1.88,13.11,35.43,35.39,35.47,11.05,0.18,0.18,0.18,0.18
1,corr,조정기,11875,2.43,15.01,45.23,45.20,45.25,11.44,0.07,0.07,0.07,0.07
2,up2,상승기②,186805,1.00,12.40,36.72,36.69,36.79,9.79,0.01,0.01,0.01,0.01
3,down,하락기,15948,1.66,11.90,48.29,48.25,48.28,7.91,0.01,0.01,0.01,0.01
4,rec,회복기,68190,0.50,11.23,31.52,31.49,31.56,7.80,0.00,0.00,0.00,0.00
5,up3,상승기③,151630,0.38,11.24,32.76,32.71,32.81,8.86,0.00,0.00,0.00,0.00


### 4.2 자치구별

![특성 결측률 — 자치구](../reports/eda/figures/10_특성_결측률_자치구.png)

### 4.3 과거 자료의 한계

오래된 단지일수록 단지 정보가 비고, 단지 내부값은 현재값을 과거에 적용한 가정이다. 입지 값은 계약 시점을 반영한다.

In [15]:
display(T('04_특성_결측_건축시기별'))
display(T('04_특성_결측_세대수출처별'))
display(T('04_현재값가정_국면별'))
display(T('04_좌표기준_국면별'))
T('04_입지_단지내_시점변화')

,건축시기,거래건수,세대수,세대당 주차대수,난방방식,복도유형,시공사(상위10 여부),사용승인일,최근접 역 거리,1km 내 노선 수,최근접 초등학교 거리,1km 내 학교 수
0,1979년 이전,13999,13.16,69.94,58.90,58.90,58.90,37.02,0.51,0.51,0.51,0.51
1,1980년대,83331,3.09,31.19,40.41,40.41,40.47,20.30,0.25,0.25,0.25,0.25
2,1990년대,164412,0.43,10.14,31.03,30.89,30.93,16.77,0.00,0.00,0.00,0.00
3,2000년대,218404,0.41,4.64,34.04,34.04,34.24,3.68,0.03,0.03,0.03,0.03
4,2010년대,111925,0.39,10.87,34.96,34.96,34.96,1.45,0.00,0.00,0.00,0.00
5,2020년 이후,25521,1.37,3.14,44.23,44.26,44.23,2.19,0.00,0.00,0.00,0.00


,세대수_출처,거래건수,세대수,세대당 주차대수,난방방식,복도유형,시공사(상위10 여부),사용승인일,최근접 역 거리,1km 내 노선 수,최근접 초등학교 거리,1km 내 학교 수
0,(세대수 없음),6795,100,100.00,98.29,98.29,98.29,98.28,5.27,5.27,5.27,5.27
1,건축물대장,196883,0,16.31,98.21,98.21,98.21,26.04,0.00,0.00,0.00,0.00
2,건축물대장(번지 합계),17592,0,38.93,99.12,99.12,99.12,11.04,0.00,0.00,0.00,0.00
3,단지 목록(K-apt),396322,0,7.52,0.06,0.00,0.14,0.00,0.00,0.00,0.00,0.00


,국면ID,국면,거래건수,현재값가정_표시,비율(%)
0,up1,상승기①,183144,118329,64.61
1,corr,조정기,11875,6507,54.80
2,up2,상승기②,186805,118267,63.31
3,down,하락기,15948,8253,51.75
4,rec,회복기,68190,46717,68.51
5,up3,상승기③,151630,102036,67.29


,국면ID,국면,(좌표 없음),단지 대표점,주소 지점(지오코딩)
0,up1,상승기①,0.18,79.09,20.72
1,corr,조정기,0.07,67.53,32.40
2,up2,상승기②,0.01,77.58,22.41
3,down,하락기,0.01,62.52,37.47
4,rec,회복기,0.00,82.85,17.15
5,up3,상승기③,0.00,81.37,18.63


,특성,좌표_있는_단지수,값이_시점에_따라_바뀐_단지,비율(%)
0,최근접 역 거리,8974,478,5.33
1,1km 내 노선 수,8974,856,9.54
2,최근접 초등학교 거리,8974,152,1.69
3,1km 내 학교 수,8974,679,7.57


### 4.4 역·학교 거리

![역·학교 거리 분포](../reports/eda/figures/11_역_학교_거리_분포.png)

In [16]:
display(T('04_거리_구간'))
T('04_입지_국면별')

,직선거리,최근접 역_거래건수,최근접 역_비율(%),최근접 초등학교_거래건수,최근접 초등학교_비율(%)
0,250m 이하,90824,14.71,224184,36.32
1,250~500m,240157,38.91,304100,49.27
2,500~750m,161789,26.21,80215,13.00
3,750m~1km,67005,10.86,8050,1.30
4,1~1.5km,41458,6.72,561,0.09
5,1.5km 초과,16001,2.59,124,0.02


,국면ID,국면,역 거리 중앙값(m),역 500m 이내(%),1km 내 노선 수 평균,초등학교 거리 중앙값(m),1km 내 학교 수 중앙값
0,up1,상승기①,491,51.51,1.64,300,10
1,corr,조정기,480,53.24,1.63,313,9
2,up2,상승기②,483,52.61,1.68,304,10
3,down,하락기,457,57.87,1.78,328,9
4,rec,회복기,460,56.85,1.81,298,10
5,up3,상승기③,465,55.55,1.78,304,10


## 5. 정리

1. 파일은 문서와 일치한다(617,592건 × 43열). 완전 중복·ID 중복 없음.
2. 국면 표본이 크게 기운다. 하락 쪽(하락기+조정기)은 전체 거래의 5% 미만이다.
3. 하락기에 거래된 집은 구성이 다르다(소형·소규모 단지 비중 증가, 거래된 단지 수 감소). 원시 가격을 국면끼리 비교하지 않는다.
4. 난방·복도·시공사 결측은 K-apt 미등록(소규모) 단지에 몰려 있고, 국면·자치구에 따라 결측률이 다르다.
5. 문서에 없던 값 문제(주차 0, 사용승인일 형식)와 문서와 다른 건수는 보고서 §1.5·§6에 정리했다.

해석과 주의 사항 전체: [`reports/eda/EDA_report_ko.md`](../reports/eda/EDA_report_ko.md)